# General stats

In [ ]:
%%configure -n default.spark -f
{
    "number_of_workers": 10,
    "session_type": "etl",
    "glue_version": "5.0",
    "worker_type": "G.1X",
    "idle_timeout": 60,
    "timeout": 600,
    "--enable-glue-datacatalog": "true",
    "--enable-auto-scaling": "true",
    "--project_s3_path": "S3_PROJECT_PATH",
    "--redshift_iam_role": "REDSHIFT_IAM_ROLE",
    "--redshift_tempdir": "REDSHIFT_TEMP_DIR",
    "--enable-lakeformation-fine-grained-access": "false"
}

In [ ]:
%%pyspark default.spark
import pyspark
from pyspark.context import SparkContext
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from itertools import chain

In [ ]:
%%pyspark default.spark
sc = SparkContext.getOrCreate()
spark = SparkSession.builder.getOrCreate()

In [ ]:
%%pyspark default.spark
df = spark.read.parquet("SOURCE_S3_PATH")

In [ ]:
%%pyspark default.spark
df.columns

An ascension 20 run in slay the spire should have around 52 - 57 runs, depending on whether or not the player enters act 4 (The game considers beating the 2 act 3 bosses a victory, even if player dies in act 4). While it is the case that winning runs end in floor 57 or 52, some runs end earlier than 52 and some more than 57. I do not know why this is the case but my hypothesis is that these runs are possibly modded. Since they make up a small minority of victory runs, I will be removing them from the dataset. I will also remove any runs above floor 57 regardless of victory


In [ ]:
%%pyspark default.spark
df.select('play_id', 'relics_obtained').head(10)[3]


In [ ]:
%%pyspark default.spark
df = df.filter(((df.victory == True) & (df.floor_reached.between(52, 57))) | ((df.victory == False) & (df.floor_reached <= 52) & (df.floor_reached > 0)))

# Create danger_taken

## Determine danger_taken weights based on health by enemy

In [ ]:
%%pyspark default.spark
dmg_taken_by_enemy = df\
    .select('play_id', f.explode('damage_taken').alias('dt'))\
    .withColumn("temp", f.lit(0))\
    .withColumns({"dt_"+c : f.col(f"dt.{c}") for c in ['damage', 'enemies']})\
    .groupBy('temp')\
    .pivot('dt_enemies')\
    .agg(f.round(f.avg('dt_damage'), 2).alias('avg_dmg_taken'))\
    .drop(*['temp', 'null'])



In [ ]:
%%pyspark default.spark

dmg_taken_dict = dmg_taken_by_enemy.toPandas().to_dict('index')[0]

In [ ]:
%%pyspark default.spark
mapping = f.create_map([f.lit(x) for x in chain(*dmg_taken_dict.items())])

## Create Column

In [ ]:
%%pyspark default.spark

df = df.withColumn(
    "danger_taken", 
    f.aggregate(
        "damage_taken.enemies",
        f.lit(0.0),
        lambda acc, x: acc + f.coalesce(mapping[x], f.lit(0.0))
    ).cast("float")
)


# General stats dataframe

In [ ]:
%%pyspark default.spark
df.select("play_id", "victory", "win_rate", "floor_reached", "danger_taken")\
    .coalesce(1).write.option("header", True).mode("overwrite").csv("GENERAL_OUTPUT_S3_PATH")


# Final Deck stats

In [ ]:
%%pyspark default.spark
#Defend_B and Strike_B presumably stands for Defend Blue and Strike Blue, the basic cards for defect
# Some cards in this list may have different names due to them using old names for the cards in the save files
# Here is a list of cards with changed name :
# - Conserve Battery -> Charge Battery
# - Gash -> Claw
# - Lockon -> Bullseye
# - Redo -> Recursion
# - Steam Power -> Steam Barrier
# - Undo -> Equilibrium
# - Steam -> Overclock (I can't find any source of this on the web, but it is the only one left that fits)

defect_cards = {
  "Defend_B": "Basic",
  "Dualcast": "Basic",
  "Strike_B": "Basic",
  "Zap": "Basic",
  "Ball Lightning": "Common",
  "Barrage": "Common",
  "Beam Cell": "Common",
  "Conserve Battery": "Common",
  "Gash": "Common",
  "Cold Snap": "Common",
  "Compile Driver": "Common",
  "Coolheaded": "Common",
  "Go for the Eyes": "Common",
  "Hologram": "Common",
  "Leap": "Common",
  "Rebound": "Common",
  "Redo": "Common",
  "Stack": "Common",
  "Steam Power": "Common",
  "Streamline": "Common",
  "Sweeping Beam": "Common",
  "Turbo": "Common",
  "Aggregate": "Uncommon",
  "Auto Shields": "Uncommon",
  "Blizzard": "Uncommon",
  "BootSequence": "Uncommon",
  "Lockon": "Uncommon",
  "Capacitor": "Uncommon",
  "Chaos": "Uncommon",
  "Chill": "Uncommon",
  "Consume": "Uncommon",
  "Darkness": "Uncommon",
  "Defragment": "Uncommon",
  "Doom and Gloom": "Uncommon",
  "Double Energy": "Uncommon",
  "Undo": "Uncommon",
  "FTL": "Uncommon",
  "Force Field": "Uncommon",
  "Fusion": "Uncommon",
  "Genetic Algorithm": "Uncommon",
  "Glacier": "Uncommon",
  "Heatsinks": "Uncommon",
  "Hello World": "Uncommon",
  "Loop": "Uncommon",
  "Melter": "Uncommon",
  "Steam": "Uncommon",
  "Recycle": "Uncommon",
  "Reinforced Body": "Uncommon",
  "Reprogram": "Uncommon",
  "Rip and Tear": "Uncommon",
  "Scrape": "Uncommon",
  "Self Repair": "Uncommon",
  "Skim": "Uncommon",
  "Static Discharge": "Uncommon",
  "Storm": "Uncommon",
  "Sunder": "Uncommon",
  "Tempest": "Uncommon",
  "White Noise": "Uncommon",
  "All For One": "Rare",
  "Amplify": "Rare",
  "Biased Cognition": "Rare",
  "Buffer": "Rare",
  "Core Surge": "Rare",
  "Creative AI": "Rare",
  "Echo Form": "Rare",
  "Electrodynamics": "Rare",
  "Fission": "Rare",
  "Hyperbeam": "Rare",
  "Machine Learning": "Rare",
  "Meteor Strike": "Rare",
  "Multi-Cast": "Rare",
  "Rainbow": "Rare",
  "Reboot": "Rare",
  "Seek": "Rare",
  "Thunder Strike": "Rare"
}

defect_card_list = list(defect_cards.keys()) + [k+"+1" for k in defect_cards.keys()]

In [ ]:
%%pyspark default.spark
final_deck_ohe = df.select('play_id', f.explode('master_deck').alias('final_deck_card'))\
    .groupBy('play_id')\
    .pivot('final_deck_card')\
    .count()\
    .na.fill(0)\
    .select("play_id", *defect_card_list)


In [ ]:
%%pyspark default.spark
final_deck_ohe.coalesce(1).write.option("header", True).mode("overwrite").csv("FINAL_DECK_OHE_OUTPUT_S3_PATH")

# Relic Stats

In [ ]:
%%pyspark default.spark

final_relics_ohe = df.select('play_id', f.explode('relics').alias('final_relic'))\
    .groupBy('play_id')\
    .pivot('final_relic')\
    .count()\
    .na.fill(0)

In [ ]:
%%pyspark default.spark
final_relics_ohe.coalesce(1).write.option('header', True).mode('overwrite').csv('FINAL_RELICS_OHE_OUTPUT_S3_PATH')


In [ ]:
%%pyspark default.spark
relic_timeline_df = df.select('play_id', f.explode('relics_obtained').alias('relics_obtained_info'))\
    .withColumns({
        'relic_obtained_floor' : f.col('relics_obtained_info.floor'),
        'relic_obtained' : f.col('relics_obtained_info.key')
    })\
    .drop('relics_obtained_info')


In [ ]:
%%pyspark default.spark
relic_timeline_df.coalesce(1).write.option('header', True).mode('overwrite').csv('RELIC_TIMELINE_OUTPUT_S3_PATH')


# Card Level stats

In [ ]:
%%pyspark default.spark
draft_df = df.select("play_id", "card_choices")\
            .select("*",f.explode("card_choices").alias("draft"))\
            .drop("card_choices")\
            .withColumns({"draft_"+c : f.col(f"draft.{c}") for c in ["floor", "not_picked", "picked"]})\
            .drop("draft")

In [ ]:
%%pyspark default.spark

card_pick_df = draft_df.select("play_id", 
                f.col("draft_floor").alias("floor_encounter"), 
                f.explode("draft_not_picked").alias("card")).withColumn("picked", f.lit(0))\
                .union(\
                    draft_df.select(
                        "play_id",
                        f.col("draft_floor").alias("floor_encounter"),
                        f.col("draft_picked").alias("card")).withColumn("picked", f.lit(1))\
                )

In [ ]:
%%pyspark default.spark
card_pick_df\
    .coalesce(1).write.option("header", True).mode("overwrite").csv("CARD_TIMELINE_OUTPUT_S3_PATH")